# 01 — Referee data pipeline

This notebook launches the existing CPU-only `referee-data` package in its separate Python 3.13 environment. It does not import the data package into this notebook kernel.

`sample` is a smoke/wiring profile. The real `train` profile needs explicit authorization and is not a smoke test. No download starts until you enable a step.


## Setup

From the repository root, sync the data project once:

```sh
uv sync --project models/referee/data --frozen
```


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import json

from referee_evaluation.dataset_pipeline import PipelineOptions, run_step

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'data').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
DATA_PROJECT = REPOSITORY_ROOT / 'models' / 'referee' / 'data'
print('Repository:', REPOSITORY_ROOT)


## Options

Enable only the steps you intend to run. `download` reads the network; `prepare` and `export` write local artifacts. The `train` profile requires separate authorization for the real source archives.


In [ ]:
PROFILE = 'sample'          # 'sample' or 'train'
DATA_ROOT = DATA_PROJECT / '.local' / 'data'
RUN_DOWNLOAD = False
RUN_PREPARE = False
RUN_PREVIEW = False
RUN_EXPORT = False
ALLOW_REAL_TRAIN = False

if PROFILE == 'train' and not ALLOW_REAL_TRAIN:
    raise ValueError('Set ALLOW_REAL_TRAIN = True only after authorization for the real train profile.')

options = PipelineOptions(data_root=DATA_ROOT, profile=PROFILE, project=DATA_PROJECT)
for step in ('download', 'prepare', 'preview', 'export'):
    print(step, '->', ' '.join(options.command(step)))


## Run selected steps

The output below is the CLI's JSON report. A nonzero exit raises with stderr, so a failure is not hidden.


In [ ]:
ENV = dict(os.environ)
ENV.pop('PYTHONPATH', None)
ENV.pop('PYTHONHOME', None)

for step, enabled in (
    ('download', RUN_DOWNLOAD),
    ('prepare', RUN_PREPARE),
    ('preview', RUN_PREVIEW),
    ('export', RUN_EXPORT),
):
    if not enabled:
        print(f'{step}: disabled')
        continue
    report = run_step(options, step, ENV)
    print(json.dumps(report, indent=2))


## Outputs

- Prepared manifests and images: `models/referee/data/.local/data/<profile>/prepared/`
- Dataset summary: `models/referee/data/.local/data/<profile>/dataset-summary.json`
- Exported trainer ZIPs and checksums: `models/referee/data/.local/data/<profile>/exports/`

Next: open `02_dataset_audit.ipynb` and audit the same profile.
